In [49]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
import tqdm
import re
import os
import glob
import spectres
from astropy import units as u

%matplotlib inline

In [90]:
SPS_HOME = os.path.abspath(os.path.join(os.getcwd(), '..'))
# SPS_HOME = os.getenv('SPS_HOME')
# SPS_HOME = SPS_HOME.replace('fsps', 'fsps_dev')  # -> I call my development folder for fsps 'fsps_dev' to keep it separate from my working fsps installation

# choose one metallicity value for this run
zfraci = 1
zstr = 'MW'

print(SPS_HOME)
print(zstr)

/Users/mreefe/Dropbox/Astrophysics/fsps_dev
MW


In [91]:
# Define the teff, logg, and logz arrays that cover the grid of UVBLUE models

teff = np.arange(15000, 62000, 1000)
logt = np.log10(teff)

logg = np.arange(2., 4.41, 0.2)
logg = np.round(logg, 1)

zfrac = np.array([1/31, 1/7, 1/2, 1.])
logz = np.round(np.log10(zfrac), 1)

print(logt)
print(logg)
print(logz)

[4.17609126 4.20411998 4.23044892 4.25527251 4.2787536  4.30103
 4.32221929 4.34242268 4.36172784 4.38021124 4.39794001 4.41497335
 4.43136376 4.44715803 4.462398   4.47712125 4.49136169 4.50514998
 4.51851394 4.53147892 4.54406804 4.5563025  4.56820172 4.5797836
 4.59106461 4.60205999 4.61278386 4.62324929 4.63346846 4.64345268
 4.65321251 4.66275783 4.67209786 4.68124124 4.69019608 4.69897
 4.70757018 4.71600334 4.72427587 4.73239376 4.74036269 4.74818803
 4.75587486 4.76342799 4.77085201 4.77815125 4.78532984]
[2.  2.2 2.4 2.6 2.8 3.  3.2 3.4 3.6 3.8 4.  4.2 4.4]
[-1.5 -0.8 -0.3  0. ]


In [92]:
# Define the wavelength grid that we will resample onto
# -> even logarithmic spacing by ~1% of the current wavelength
w_1 = 90. * 1.01**np.arange(0, int(np.log(600/90)/np.log(1.01))) 
# -> finer sampling in the UV/optical; regions of interest
w_2 = np.arange(600., 1800., 0.1)
w_3 = np.arange(1800., 2000., 0.2)
w_4 = np.arange(2000., 3000., 0.5)
w_5 = np.arange(3000., 9000., 1.)
# -> spacing by ~0.3% of the current wavelength
w_6 = 9000. * 1.003**np.arange(0, int(np.log(9.99e6/9000)/np.log(1.003))) 
wavelength = np.concatenate((w_1, w_2, w_3, w_4, w_5, w_6))

print('length = ', len(wavelength))
print(wavelength)

length =  23530
[9.00000000e+01 9.09000000e+01 9.18090000e+01 ... 9.87467880e+06
 9.90430284e+06 9.93401575e+06]


In [93]:
# wavelength must be in angstroms!
def airtovac(wavelength):
    # see: https://www.astro.uu.se/valdwiki/Air-to-vacuum%20conversion
    s = 1e4 / wavelength
    n = 1 + 0.00008336624212083 + 0.02408926869968 / (130.1065924522 - s**2) + 0.0001599740894897 / (38.92568793293 - s**2)
    # do not alter wavelengths below 2000 angstroms 
    wh = np.where(wavelength < 2000.)[0]
    n[wh] = 1.0
    return wavelength * n

In [94]:
c_ang = 299792458e10

# allocate a buffer for all of the spectra at one metallicity
library_in = np.zeros((len(wavelength), len(logt), len(logg)))

folder = f'/Users/mreefe/Dropbox/Astrophysics/stellar_templates/PoWR/Z_{zstr}/'
sed_files = glob.glob(os.path.join(folder, 'sed', '*_sed.txt'))
opt_files = glob.glob(os.path.join(folder, 'line-opt', '*_line_calib.txt'))
uv_files = glob.glob(os.path.join(folder, 'line-uv', '*_line_calib.txt'))

sed_files.sort()
opt_files.sort()
uv_files.sort()

for sed_path, opt_path, uv_path in tqdm.tqdm(zip(sed_files, opt_files, uv_files), total=len(sed_files)):

    # parse the file name to get the temp, logg, and logz
    fname = os.path.basename(sed_path)
    m = re.search(r'\_([0-9]+)\-([0-9]+)\_sed\.txt$', fname)
    teff_v = int(m.group(1))*1000
    logg_v = round(float(m.group(2))/10, 1)

    # find the indices corresponding to these values in the array
    logt_i = np.where(teff == teff_v)[0][0]
    logg_i = np.where(logg == logg_v)[0][0]

    # print(f'Teff = {teff_v:.0f} (index = {logt_i:.0f})')
    # print(f'logg = {logg_v:.1f} (index = {logg_i:.0f})')
    # print(f'logZ = {logz_v:.1f}')

    # read in the text files
    loglam_sed, logflam_sed = np.loadtxt(sed_path, unpack=True)
    lam_opt, logf_opt = np.loadtxt(opt_path, unpack=True)
    lam_uv,  logf_uv  = np.loadtxt(uv_path,  unpack=True)

    # handle the -100's 
    good = np.where(logflam_sed != -100.)[0]
    loglam_sed = loglam_sed[good]
    logflam_sed = logflam_sed[good]

    # convert from log
    lam_sed = 10**loglam_sed
    f_sed = 10**logflam_sed
    f_opt = 10**logf_opt
    f_uv  = 10**logf_uv

    # renormalize the high-res spectra to smoothly join with the SEDs
    wh_opt = (lam_sed > lam_opt.min()) & (lam_sed < lam_opt.max())
    wh_uv  = (lam_sed > lam_uv.min()) & (lam_sed < lam_uv.max())
    f_opt *= np.nanmedian(f_sed[wh_opt])/np.nanmedian(f_opt)
    f_uv  *= np.nanmedian(f_sed[wh_uv])/np.nanmedian(f_uv)

    # remove the overlapping points from the SED
    good = np.where(~(wh_opt|wh_uv))[0]
    
    # smoothly join
    wave = np.concatenate((lam_sed[good], lam_uv, lam_opt)) 
    flam = np.concatenate((f_sed[good], f_uv, f_opt))
    ss = np.argsort(wave)
    wave = wave[ss]
    flam = flam[ss]

    # Convert the units
    fnu = flam*wave**2/c_ang

    # normalize to integrate to 1
    norm = np.trapz(flam, wave)
    fnu /= norm

    # perform the flux-conserving resampling onto the output wavelength grid
    # flux_o = spectres.spectres(wavelength, wave_i, flux_i, fill=0., verbose=False)
    fnu_o = np.interp(wavelength, wave, fnu, left=0., right=0.)

    # insert it into the 3D array
    library_in[:, logt_i, logg_i] = fnu_o

    # # plot the new and old spectrum to compare them
    # fig, ax = plt.subplots()
    # ax.plot(wave, fnu)
    # ax.plot(wavelength, fnu_o)
    # ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlabel('Wavelength (angstrom)')
    # ax.set_ylabel('Flambda')
    # # ax.set_xlim(900, 1800)
    # # ax.set_ylim(flux_i[(wave_i > 900) & (wave_i < 1800)].min()*0.98, flux_i[(wave_i > 900) & (wave_i < 1800)].max()*1.02)
    # plt.show()
    # plt.close()


100%|██████████| 243/243 [00:12<00:00, 19.85it/s]


In [95]:
# Read in WMBasic templates for comparison
wmb_logt = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/WMBASIC.teff'))
wmb_logg = np.array([3.5, 4., 4.5])
wmb = np.loadtxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/WMBASIC_z0.0010.spec'))
w_wmb = wmb[:,0]
wmb = wmb[:,1:]
wmb = wmb.reshape(wmb.shape[0], len(wmb_logg), len(wmb_logt))

template_folder = os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/PoWR/template_plots')
if not os.path.exists(template_folder):
    os.makedirs(template_folder)

for j in range(len(wmb_logg)):
    for i in range(len(wmb_logt)):
        jj = np.nanargmin(np.abs(wmb_logg[j] - logg))
        ii = np.nanargmin(np.abs(wmb_logt[i] - logt))
        fig, ax = plt.subplots()
        ax.plot(w_wmb, wmb[:,j,i], label='WMBASIC')
        ax.plot(wavelength, library_in[:,ii,jj], label='PoWR')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel(r'Wavelength ($\mathring{\rm A}}$)')
        ax.set_ylabel(r'$F_\nu$ (normalized)')
        ax.set_xlim(900, 1800)
        ax.set_ylim(1e-22, 1e-14)
        ax.set_title(f'logg={wmb_logg[j]}, teff={10**wmb_logt[i]}')
        ax.legend()
        plt.savefig(os.path.join(template_folder, f'{j}_{i}.pdf'), dpi=300, bbox_inches='tight')
        plt.close()

In [96]:
library_in.shape

(23530, 47, 13)

In [97]:
# Save as a text file with the same format as the format 
zsun = 0.0134
z = zfraci * zsun
# need to flatten the array to 2D matching the shape of the WM-Basic arrays FSPS uses
# (flatten logt and logg axis, should be ordered as [t1_g1 t2_g1 t3_g1 ... t1_g2 t2_g2 t3_g2 ...])
library_out = library_in.reshape(library_in.shape[0], len(logg)*len(logt))
# append the wavelength column
library_out = np.concatenate((wavelength.reshape(len(wavelength), 1), library_out), axis=1)

np.savetxt(os.path.join(SPS_HOME, f'SPECTRA/Hot_spectra/PoWR/PoWR_z{z:.4f}.spec'), library_out, fmt='%.4e', delimiter=' ')

In [98]:
# save other ancillary files
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/PoWR/PoWR.teff'), logt, fmt='%13.5f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/PoWR/PoWR.logg'), logg, fmt='%13.5f')
np.savetxt(os.path.join(SPS_HOME, 'SPECTRA/Hot_spectra/PoWR/PoWR_zlegend.dat'), zfrac*zsun, fmt='%7.4f')